# Export and reporting campaign
Runs the remaining export/final/report stages and emits assistant and thesis packs.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("EDGEGUARD_PROJECT_ROOT", ".")).resolve()
CAMPAIGN_ROOT = Path(os.environ.get("EDGEGUARD_CAMPAIGN_ROOT", "/content/edgeguard-campaign"))
PROJECT_COMMIT = os.environ.get("EDGEGUARD_PROJECT_COMMIT", "")
AUTO_CONTINUE = os.environ.get("EDGEGUARD_AUTO_CONTINUE", "0") == "1"
actual = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if len(PROJECT_COMMIT) != 40 or actual != PROJECT_COMMIT:
    raise RuntimeError("exact project commit mismatch")
if not (CAMPAIGN_ROOT / "campaign_manifest.json").is_file():
    raise RuntimeError("campaign state is missing")
base = [sys.executable, "-m", "edgeguard.campaign", "--repository", str(PROJECT_ROOT)]

In [ ]:
plan = subprocess.run(
    base + ["plan", "--campaign-root", str(CAMPAIGN_ROOT)],
    check=True,
    capture_output=True,
    text=True,
)
print(json.dumps(json.loads(plan.stdout), indent=2))
if AUTO_CONTINUE:
    subprocess.run(base + ["resume", "--campaign-root", str(CAMPAIGN_ROOT)], check=True)
subprocess.run(
    base + ["report", "--campaign-root", str(CAMPAIGN_ROOT), "--audience", "assistant"], check=True
)
subprocess.run(
    base + ["report", "--campaign-root", str(CAMPAIGN_ROOT), "--audience", "thesis"], check=True
)